# Apple music store — revenue questions in SQL

Chinook is a sample music-store database. This notebook answers three business questions against it, in SQL first, and then brings the answers into pandas.

**Ticket 1 — the map.** Before writing any query I need to know how the tables fit together and what one row of each table means. Nothing here answers a business question yet.

In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect('data/Chinook_Sqlite.sqlite')
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


## Every table, and what one row means

| table | one row is |
|---|---|
| `Artist` | one artist |
| `Album` | one album, belonging to one artist |
| `Track` | one track, on one album, of one genre and media type |
| `Genre`, `MediaType` | lookup rows |
| `Playlist`, `PlaylistTrack` | a playlist, and one track's membership in one playlist |
| `Customer` | one customer, with a country, and the employee who supports them |
| `Employee` | one employee; `ReportsTo` points at their manager in the same table |
| `Invoice` | **one sale** — one customer, one date, one billing address, one total |
| `InvoiceLine` | **one track on one invoice** — unit price and quantity |

## Foreign keys, read from the database rather than guessed

SQLite stores the declared foreign keys. Reading them out is more reliable than inferring joins from column names.

In [2]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con).name
rows = []
for t in tables:
    for fk in con.execute(f'PRAGMA foreign_key_list("{t}")'):
        rows.append({'from_table': t, 'from_column': fk[3], 'to_table': fk[2], 'to_column': fk[4]})
fks = pd.DataFrame(rows).sort_values(['from_table','from_column']).reset_index(drop=True)
fks

,from_table,from_column,to_table,to_column
0,Album,ArtistId,Artist,ArtistId
1,Customer,SupportRepId,Employee,EmployeeId
2,Employee,ReportsTo,Employee,EmployeeId
3,Invoice,CustomerId,Customer,CustomerId
4,InvoiceLine,InvoiceId,Invoice,InvoiceId
5,InvoiceLine,TrackId,Track,TrackId
6,PlaylistTrack,PlaylistId,Playlist,PlaylistId
7,PlaylistTrack,TrackId,Track,TrackId
8,Track,AlbumId,Album,AlbumId
9,Track,GenreId,Genre,GenreId


## The two join paths every later ticket needs

### Track → the country it was sold in

```
Track.TrackId  →  InvoiceLine.TrackId
InvoiceLine.InvoiceId  →  Invoice.InvoiceId
Invoice.CustomerId  →  Customer.CustomerId
```

That is **three hops**. There are two countries on the way: `Invoice.BillingCountry` (where this sale was billed) and `Customer.Country` (where the customer lives). They are usually the same but are separate columns, and ticket 2 has to say which one it uses.

### Track → the employee who owns that customer

```
… → Customer.SupportRepId  →  Employee.EmployeeId
```

One more hop past `Customer`. **`SupportRepId` is the only link from a sale to an employee** — there is no employee on `Invoice` itself, so "top-performing sales employee" can only mean *the rep whose customers spent the most*.

## Grain — the thing that decides whether a total is right

- **`InvoiceLine` is one row per track per invoice.** Counting rows here counts *tracks sold*; summing `UnitPrice * Quantity` here gives revenue per track.
- **`Invoice` is one row per sale.** Counting rows here counts *orders*; summing `Total` here gives revenue per invoice — and `Total` already equals the sum of that invoice's lines.

Why this matters: joining `Invoice` to `InvoiceLine` and then summing `Invoice.Total` counts each invoice's total **once per line on it** — an invoice with five tracks is counted five times. Revenue must be summed from `InvoiceLine`, or from `Invoice` alone, never from `Invoice.Total` after a join to lines.

In [3]:
# Sanity check on the grain claim: Invoice.Total really is the sum of its lines.
pd.read_sql_query('''
SELECT COUNT(*) AS invoices,
       SUM(CASE WHEN ROUND(i.Total, 2) = ROUND(l.line_total, 2) THEN 1 ELSE 0 END) AS totals_match_lines
FROM Invoice i
JOIN (SELECT InvoiceId, SUM(UnitPrice * Quantity) AS line_total FROM InvoiceLine GROUP BY InvoiceId) l
  ON l.InvoiceId = i.InvoiceId
''', con)

,invoices,totals_match_lines
0,412,412


# Ticket 2 — the three business questions, answered in SQL

One query per question. Each joins, groups and orders **in the database** and returns the answer, not the raw rows. Under each, one sentence saying what the number counts.

## 1. The ten best-selling tracks

**"Best-selling" here means units sold**, not revenue. A track is *sold* when a customer buys it; revenue mixes that with price, and in this store almost everything is \$0.99 except video tracks at \$1.99 — so a revenue ranking is really "which \$1.99 videos sold", a pricing fact rather than a popularity fact. Both are shown so the difference is visible.

⚠️ **A caveat the query cannot hide: no track in this store has sold more than 2 units.** The "top ten" is a tie among well over a hundred tracks at 2 units, broken alphabetically. The honest answer to "what are the best-selling tracks" in this dataset is *there are none — sales are flat.* That is worth more to the business than a list.

In [4]:
best_by_units = pd.read_sql_query('''
SELECT t.Name AS track, ar.Name AS artist, SUM(l.Quantity) AS units_sold
FROM InvoiceLine l
JOIN Track  t  ON t.TrackId  = l.TrackId
JOIN Album  al ON al.AlbumId = t.AlbumId
JOIN Artist ar ON ar.ArtistId = al.ArtistId
GROUP BY t.TrackId
ORDER BY units_sold DESC, t.Name
LIMIT 10
''', con)
best_by_units

,track,artist,units_sold
0,A Cor Do Sol,Cidade Negra,2
1,A Melhor Forma,Titãs,2
2,A Novidade (Live),Gilberto Gil,2
3,"Abraham, Martin And John",Marvin Gaye,2
4,Aces High,Iron Maiden,2
5,All Along The Watchtower,U2,2
6,Amor De Muito,Chico Science & Nação Zumbi,2
7,Ando Meio Desligado,Os Mutantes,2
8,As Rosas Não Falam (Beth Carvalho),Various Artists,2
9,Azul,Djavan,2


*Counts:* units of each track across every invoice line, summed per track.

In [5]:
# How many tracks share the top spot? This is the number that makes the list above honest.
pd.read_sql_query('''
SELECT units_sold, COUNT(*) AS tracks_at_this_level
FROM (SELECT TrackId, SUM(Quantity) AS units_sold FROM InvoiceLine GROUP BY TrackId)
GROUP BY units_sold ORDER BY units_sold DESC
''', con)

,units_sold,tracks_at_this_level
0,2,256
1,1,1728


In [6]:
best_by_revenue = pd.read_sql_query('''
SELECT t.Name AS track, t.UnitPrice AS price, ROUND(SUM(l.UnitPrice * l.Quantity), 2) AS revenue
FROM InvoiceLine l
JOIN Track t ON t.TrackId = l.TrackId
GROUP BY t.TrackId
ORDER BY revenue DESC, t.Name
LIMIT 10
''', con)
best_by_revenue

,track,price,revenue
0,Gay Witch Hunt,1.99,3.98
1,Hot Girl,1.99,3.98
2,How to Stop an Exploding Man,1.99,3.98
3,Phyllis's Wedding,1.99,3.98
4,Pilot,1.99,3.98
5,The Fix,1.99,3.98
6,The Woman King,1.99,3.98
7,Walkabout,1.99,3.98
8,"""?""",1.99,1.99
9,...And Found,1.99,1.99


*Counts:* `UnitPrice × Quantity` from the invoice **line** (what was actually charged), summed per track. Every row is a \$1.99 track sold twice — the revenue list is the price list in disguise.

## 2. Which country generates the most revenue

**Using `Invoice.BillingCountry`**, not `Customer.Country`. The question is where the *money* came from, and billing country is a property of the sale; a customer who moves countries keeps their old invoices where they were billed. In this dataset the two agree on every invoice (checked below), so the choice does not change the answer here — but it would on real data, and stating it is the point.

In [7]:
revenue_by_country = pd.read_sql_query('''
SELECT i.BillingCountry AS country,
       ROUND(SUM(l.UnitPrice * l.Quantity), 2) AS revenue,
       COUNT(DISTINCT i.InvoiceId) AS invoices
FROM InvoiceLine l
JOIN Invoice i ON i.InvoiceId = l.InvoiceId
GROUP BY i.BillingCountry
ORDER BY revenue DESC
''', con)
revenue_by_country.head(10)

,country,revenue,invoices
0,USA,523.06,91
1,Canada,303.96,56
2,France,195.10,35
3,Brazil,190.10,35
4,Germany,156.48,28
5,United Kingdom,112.86,21
6,Czech Republic,90.24,14
7,Portugal,77.24,14
8,India,75.26,13
9,Chile,46.62,7


In [8]:
# Does billing country ever differ from the customer's country? If it never does, the choice above is moot HERE.
pd.read_sql_query('''
SELECT COUNT(*) AS invoices_where_billing_differs_from_customer_country
FROM Invoice i JOIN Customer c ON c.CustomerId = i.CustomerId
WHERE i.BillingCountry <> c.Country
''', con)

,invoices_where_billing_differs_from_customer_country
0,0


*Counts:* revenue from invoice lines, grouped by the country on the invoice. Summed from `InvoiceLine`, not `Invoice.Total`, so no invoice is counted once per line (see the grain note in ticket 1). **USA leads at \$523.06 across 91 invoices; Canada is second at \$303.96.**

## 3. The top-performing sales employee

Through `Customer.SupportRepId` — the only link from a sale to a person. "Top-performing" is defined as **revenue from the customers they support**. Customers-per-rep is shown beside it, because a rep with more customers should be expected to bring more revenue; the fairer comparison is revenue per customer.

In [9]:
revenue_by_rep = pd.read_sql_query('''
SELECT e.FirstName || ' ' || e.LastName AS employee,
       e.Title,
       ROUND(SUM(l.UnitPrice * l.Quantity), 2) AS revenue,
       COUNT(DISTINCT c.CustomerId) AS customers,
       ROUND(SUM(l.UnitPrice * l.Quantity) / COUNT(DISTINCT c.CustomerId), 2) AS revenue_per_customer
FROM InvoiceLine l
JOIN Invoice  i ON i.InvoiceId  = l.InvoiceId
JOIN Customer c ON c.CustomerId = i.CustomerId
JOIN Employee e ON e.EmployeeId = c.SupportRepId
GROUP BY e.EmployeeId
ORDER BY revenue DESC
''', con)
revenue_by_rep

,employee,Title,revenue,customers,revenue_per_customer
0,Jane Peacock,Sales Support Agent,833.04,21,39.67
1,Margaret Park,Sales Support Agent,775.40,20,38.77
2,Steve Johnson,Sales Support Agent,720.16,18,40.01


*Counts:* revenue from every invoice line, attributed to the support rep of the customer on that invoice. **Jane Peacock leads at \$833.04 — but she also has the most customers (21); per customer the three reps are within \$1 of each other.** Only three employees are reps at all; the other five have no customers and do not appear.